# See this before moving ahead

In [1]:
from typing import TypedDict

In [2]:
class Person(TypedDict):
    name: str
    age: int

In [6]:
person1: Person = {'name': 'Nikunj', 'age': 24}

In [9]:
a = Person(
    {
        'name': 'Nikunj',
        'age': 24
    }
)

# Problem statement:
We have some reviwes on movies. We will pass these reviews to LLMs and we want a structured output in the form of dictionary. <br>
dict = {`summary`: `sumary of review, ``sentiment`: Positive, negative or neutral}

1) `Simple Typeddict`

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict

In [3]:
load_dotenv()

True

In [24]:
model = ChatOpenAI(
    #model = 'gpt-3.5-turbo',
    temperature = 0.3,
    max_tokens = 300
)

In [25]:
# Schema

class Review(TypedDict):
    summary: str
    sentiment: str

In [26]:
structured_model = model.with_structured_output(schema = Review)


C:\Users\jugal\langchain-env\Lib\site-packages\langchain_openai\chat_models\base.py:1669: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [27]:
response = structured_model.invoke('The moviw was very bad. Story was stupid. Wastage of money and specially, time')

In [28]:
print(response)
print()

print(response['summary'])
print()

print(response['sentiment'])

{'summary': 'The movie was very bad. Story was stupid. Wastage of money and specially, time', 'sentiment': 'negative'}

The movie was very bad. Story was stupid. Wastage of money and specially, time

negative


2) Annotated TypedDict:

LLM is trained in such a manner that from Review class, just reading 'summary' and 'sentiment', it understood that it has to generate Summary and sentiment. But sometimes it may happen that the model may not be able to understand properly what it has to do. To solve this, we can add some description regarding what it has to do. This is called `Annotation`.

In [18]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional

In [12]:
load_dotenv()

True

In [33]:
model = ChatOpenAI(
    #model = 'gpt-3.5-turbo',
    #temperature = 0.5,
    #max_tokens = 30
)

In [ ]:
# Creating a dictionary schema

class Review(TypedDict):
    summary: Annotated[str, 'A brief summary of the review']
    sentiment: Annotated[str, "Sentiment of the review either Negative, Positive or Neutral"]
    key_themes: Annotated[list, 'Mention all the key topics discussed in the review']
    pros: Annotated[Optional[list[str]], 'Mention all the pros']
    cons: Annotated[Optional[list[str]], 'Mention all the cons']

# Optional means if Pros or cons are mentioned in the review, inset them in the dictionary, else ignore


In [35]:
structured_model = model.with_structured_output(schema = Review)

C:\Users\jugal\langchain-env\Lib\site-packages\langchain_openai\chat_models\base.py:1669: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [49]:
rev1 = """The Galaxy S23 is a compact yet powerful flagship smartphone.
It offers a premium design, vibrant AMOLED display, and top-tier performance 
thanks to the Snapdragon 8 Gen 2 chip. The phone handles multitasking, gaming, 
and photography smoothly, while the battery comfortably lasts a full day. 
It’s ideal for users who prefer a sleek, one-hand-friendly device without compromising on features.

Pros:
Premium and compact design
Excellent performance with Snapdragon 8 Gen 2
Stunning AMOLED display with 120Hz refresh rate
Great camera performance (especially night mode & video)
Fast charging and full-day battery life
Smooth gaming with minimal heating

Cons:
No charger included in the box
Slightly smaller battery compared to other flagships
Mono speaker sound could be better (if applicable)
Premium price for a small-sized phone
"""

response = structured_model.invoke(rev1)

In [50]:
print(response)
print()

print(response['summary'])
print()

print(response['key_themes'])
print()

print(response['pros'])
print()

print(response['cons'])
print()

print(response['sentiment'])
print()

# If we remove cons from the review, it will not add cons bcz it has Optional

{'summary': 'A compact yet powerful flagship smartphone with premium design, vibrant AMOLED display, and top-tier performance. Ideal for users who prefer a sleek, one-hand-friendly device without compromising on features.', 'sentiment': 'Positive', 'key_themes': ['Compact Design', 'AMOLED Display', 'Snapdragon 8 Gen 2 Chip', 'Battery Life', 'Gaming', 'Photography'], 'pros': ['Premium and compact design', 'Excellent performance with Snapdragon 8 Gen 2', 'Stunning AMOLED display with 120Hz refresh rate', 'Great camera performance (especially night mode & video)', 'Fast charging and full-day battery life', 'Smooth gaming with minimal heating'], 'cons': ['No charger included in the box', 'Slightly smaller battery compared to other flagships', 'Mono speaker sound could be better (if applicable)', 'Premium price for a small-sized phone']}

A compact yet powerful flagship smartphone with premium design, vibrant AMOLED display, and top-tier performance. Ideal for users who prefer a sleek, one-

In [51]:
response.keys()

dict_keys(['summary', 'sentiment', 'key_themes', 'pros', 'cons'])

# We know that TypedDict only suggests that particular variable should be int or str or list etc... But it doesn't force. We are allowed to enter values of any dtypes. But what if we want to restrict the user to enter other dtypes instead of the allowed one?

# In that case we have Pydantic. Pydantic has facility of validation and many other features.
# Let's see it in Pydantic notebook